# =========================================================
# Multi-Planar U-Net for Medical Image Segmentation (Colab)
# Author: Alan (FYP Medical Image Segmentation)
# =========================================================

In [1]:
!pip install torch torchvision matplotlib nibabel --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 30.0 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 1. Synthetic 3D Dataset (Simulating CT Volumes)
# ---------------------------------------------------------

In [3]:
class SyntheticCTDataset(Dataset):
    """
    Generates synthetic 3D CT scans (random blobs).
    Each scan is (Depth, Height, Width).
    Segmentation task: detect blobs as foreground.
    """
    def __init__(self, num_samples=50, shape=(64, 64, 64)):
        self.num_samples = num_samples
        self.shape = shape

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        vol = np.zeros(self.shape, dtype=np.float32)
        mask = np.zeros_like(vol)

        # Place random spherical "tumor"
        cx, cy, cz = np.random.randint(15, 50, size=3)
        r = np.random.randint(5, 10)
        for x in range(self.shape[0]):
            for y in range(self.shape[1]):
                for z in range(self.shape[2]):
                    if (x - cx) ** 2 + (y - cy) ** 2 + (z - cz) ** 2 < r ** 2:
                        vol[x, y, z] = 1.0
                        mask[x, y, z] = 1.0

        vol = torch.tensor(vol).unsqueeze(0)  # (1, D, H, W)
        mask = torch.tensor(mask).unsqueeze(0)
        return vol, mask

# ---------------------------------------------------------
# 2. Simple 2D U-Net (Reused from Notebook 1)
# ---------------------------------------------------------

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(64, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = DoubleConv(64, 32)
        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d1 = self.up1(b)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        return torch.sigmoid(self.final(d2))


# ---------------------------------------------------------
# 3. Multi-Planar Fusion Model
# ---------------------------------------------------------

In [5]:
class MultiPlanarUNet(nn.Module):
    """
    Processes three orthogonal views (axial, coronal, sagittal)
    and fuses their predictions.
    """
    def __init__(self):
        super().__init__()
        self.unet_axial = UNet2D()
        self.unet_coronal = UNet2D()
        self.unet_sagittal = UNet2D()
        self.fusion = nn.Conv3d(3, 1, kernel_size=1)

    def forward(self, x):
        # x: (B, 1, D, H, W)
        B, C, D, H, W = x.shape

        # Axial (slices along depth)
        axial_preds = []
        for i in range(D):
            slice2d = x[:, :, i, :, :]  # (B, 1, H, W)
            p = self.unet_axial(slice2d)
            axial_preds.append(p.unsqueeze(2))  # Add depth dim
        axial_pred = torch.cat(axial_preds, dim=2)  # (B,1,D,H,W)

        # Coronal (slices along height)
        coronal_preds = []
        for i in range(H):
            slice2d = x[:, :, :, i, :]  # (B,1,D,W)
            p = self.unet_coronal(slice2d)
            coronal_preds.append(p.unsqueeze(3))
        coronal_pred = torch.cat(coronal_preds, dim=3)

        # Sagittal (slices along width)
        sagittal_preds = []
        for i in range(W):
            slice2d = x[:, :, :, :, i]  # (B,1,D,H)
            p = self.unet_sagittal(slice2d)
            sagittal_preds.append(p.unsqueeze(4))
        sagittal_pred = torch.cat(sagittal_preds, dim=4)

        # Fuse predictions
        preds = torch.cat([axial_pred, coronal_pred, sagittal_pred], dim=1)  # (B,3,D,H,W)
        out = torch.sigmoid(self.fusion(preds))
        return out


# ---------------------------------------------------------
# 4. Training Loop (Simplified)
# ---------------------------------------------------------

In [6]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for vol, mask in loader:
        vol, mask = vol.to(device), mask.to(device)
        optimizer.zero_grad()
        preds = model(vol)
        loss = criterion(preds, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)



# ---------------------------------------------------------
# 5. Run Training
# ---------------------------------------------------------

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = SyntheticCTDataset(num_samples=10)
    loader = DataLoader(dataset, batch_size=1, shuffle=True)

    model = MultiPlanarUNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    for epoch in range(2):  # only 2 epochs for demo
        loss = train(model, loader, optimizer, criterion, device)
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

    # Show one slice prediction
    vol, mask = dataset[0]
    vol = vol.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(vol).cpu().squeeze().numpy()

    plt.subplot(1,3,1)
    plt.imshow(vol.cpu().squeeze()[32,:,:], cmap="gray")
    plt.title("CT Slice")
    plt.subplot(1,3,2)
    plt.imshow(mask.squeeze()[32,:,:], cmap="gray")
    plt.title("Ground Truth")
    plt.subplot(1,3,3)
    plt.imshow(pred[32,:,:]>0.5, cmap="gray")
    plt.title("Prediction")
    plt.show()

if __name__ == "__main__":
    main()